# BERTopic Vietnamese — Kaggle Dual T4 GPU

**Setup:** 2x NVIDIA T4 (16GB VRAM mỗi con)  
**Model:** vinai/phobert-base + UMAP + HDBSCAN  
**OOM protection:** batch embedding + memory monitor + gradient-free inference

> Đảm bảo Kaggle Accelerator = **GPU T4 x2** trước khi chạy

## 0. Cài đặt dependencies

In [ ]:
%%capture
!pip install bertopic sentence-transformers umap-learn hdbscan gensim --quiet

## 1. Imports & GPU Setup

In [ ]:
import os
import gc
import csv
import json
import pickle
import warnings
from datetime import datetime, timezone
from typing import List, Tuple, Optional, Dict, Any

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel

warnings.filterwarnings('ignore')

# ── GPU info ──────────────────────────────────────────────────────────────────
N_GPUS = torch.cuda.device_count()
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
print(f"GPUs detected   : {N_GPUS}")
for i in range(N_GPUS):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name}  |  VRAM: {props.total_memory / 1e9:.1f} GB")

## 2. Memory utilities

In [ ]:
def gpu_memory_summary():
    """In trạng thái VRAM của từng GPU."""
    for i in range(N_GPUS):
        alloc  = torch.cuda.memory_allocated(i)  / 1e9
        reserv = torch.cuda.memory_reserved(i)   / 1e9
        total  = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f"  GPU {i}: used={alloc:.2f}GB  reserved={reserv:.2f}GB  total={total:.1f}GB")


def free_gpu_memory():
    """Giải phóng cache GPU + chạy GC."""
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()


def get_free_vram_gb(device: int = 0) -> float:
    """Trả về VRAM còn trống (GB) của GPU `device`."""
    free, _ = torch.cuda.mem_get_info(device)
    return free / 1e9


print("GPU memory before loading model:")
gpu_memory_summary()

## 3. Dual-T4 BERTopic Model

In [ ]:
class DualGPUEmbedder:
    """
    Wrapper encode documents với SentenceTransformer trên 2 GPU T4.

    Chiến lược:
        - DataParallel KHÔNG hoạt động tốt với SentenceTransformer
          vì nó dùng mean-pooling sau attention → dùng manual data-split thay thế.
        - Tách batch thành 2 phần đều nhau, encode song song trên GPU 0 & 1,
          sau đó ghép lại (torch.cat).
        - OOM protection: nếu batch_size gây OOM → tự giảm 50% và retry.
    """

    def __init__(
        self,
        model_name: str = "vinai/phobert-base",
        batch_size: int = 64,
        max_seq_length: int = 256,
        verbose: bool = True,
    ):
        self.model_name    = model_name
        self.batch_size    = batch_size
        self.max_seq_length = max_seq_length
        self.verbose       = verbose
        self.n_gpus        = torch.cuda.device_count()

        if self.verbose:
            print(f"[DualGPUEmbedder] Loading {model_name} ...")

        # GPU 0 là primary; GPU 1 là secondary
        self.model_gpu0 = SentenceTransformer(model_name, device="cuda:0")
        self.model_gpu0.max_seq_length = max_seq_length

        if self.n_gpus >= 2:
            self.model_gpu1 = SentenceTransformer(model_name, device="cuda:1")
            self.model_gpu1.max_seq_length = max_seq_length
            if self.verbose:
                print("  ✅ Model loaded on GPU 0 AND GPU 1")
        else:
            self.model_gpu1 = None
            if self.verbose:
                print("  ⚠️  Only 1 GPU found — running on GPU 0 only")

    def encode(
        self,
        sentences: List[str],
        batch_size: Optional[int] = None,
        show_progress_bar: bool = True,
    ) -> np.ndarray:
        """
        Encode list of sentences → numpy array (n_docs, hidden_dim).

        OOM protection:
            Nếu VRAM còn < 2 GB trước khi encode → giảm batch_size 50%.
            Nếu vẫn OOM → giảm thêm, retry tối đa 3 lần.
        """
        if batch_size is None:
            batch_size = self.batch_size

        # Auto-reduce batch_size nếu VRAM sắp hết
        free_vram = get_free_vram_gb(0)
        if free_vram < 3.0:
            old_bs = batch_size
            batch_size = max(8, batch_size // 2)
            if self.verbose:
                print(f"  ⚠️  Low VRAM ({free_vram:.1f}GB free) → batch_size {old_bs}→{batch_size}")

        # Nếu có 2 GPU → split docs đều 2 phần, encode song song
        if self.n_gpus >= 2 and self.model_gpu1 is not None:
            return self._encode_dual_gpu(sentences, batch_size, show_progress_bar)
        else:
            return self._encode_single_gpu(self.model_gpu0, sentences, batch_size, show_progress_bar)

    def _encode_dual_gpu(
        self,
        sentences: List[str],
        batch_size: int,
        show_progress_bar: bool,
    ) -> np.ndarray:
        """Chia đôi dataset, encode song song trên GPU 0 & 1, ghép lại."""
        mid = len(sentences) // 2
        docs_0 = sentences[:mid]
        docs_1 = sentences[mid:]

        if self.verbose:
            print(f"  → GPU 0: {len(docs_0)} docs | GPU 1: {len(docs_1)} docs")

        # Encode với OOM retry
        emb_0 = self._encode_with_oom_retry(
            self.model_gpu0, docs_0, batch_size, show_progress_bar, gpu_id=0
        )
        free_gpu_memory()

        emb_1 = self._encode_with_oom_retry(
            self.model_gpu1, docs_1, batch_size, show_progress_bar, gpu_id=1
        )
        free_gpu_memory()

        return np.vstack([emb_0, emb_1])

    def _encode_single_gpu(
        self,
        model: SentenceTransformer,
        sentences: List[str],
        batch_size: int,
        show_progress_bar: bool,
    ) -> np.ndarray:
        return self._encode_with_oom_retry(
            model, sentences, batch_size, show_progress_bar, gpu_id=0
        )

    @torch.no_grad()
    def _encode_with_oom_retry(
        self,
        model: SentenceTransformer,
        sentences: List[str],
        batch_size: int,
        show_progress_bar: bool,
        gpu_id: int = 0,
        max_retries: int = 3,
    ) -> np.ndarray:
        """Encode với tối đa `max_retries` lần retry khi gặp OOM."""
        for attempt in range(max_retries):
            try:
                embeddings = model.encode(
                    sentences,
                    batch_size=batch_size,
                    show_progress_bar=show_progress_bar,
                    convert_to_numpy=True,
                    normalize_embeddings=False,
                )
                return embeddings
            except torch.cuda.OutOfMemoryError:
                free_gpu_memory()
                old_bs = batch_size
                batch_size = max(4, batch_size // 2)
                print(
                    f"  ⚠️  OOM on GPU {gpu_id} (attempt {attempt+1}/{max_retries})"
                    f" — batch_size {old_bs} → {batch_size}"
                )
                if attempt == max_retries - 1:
                    raise RuntimeError(
                        f"GPU {gpu_id} OOM sau {max_retries} lần retry. "
                        f"Thử giảm max_seq_length hoặc dùng ít documents hơn."
                    )

    # BERTopic gọi .encode() → interface tương thích
    def __call__(self, sentences, **kwargs):
        return self.encode(sentences, **kwargs)

In [ ]:
class VietnameseBERTopicModel:
    """
    BERTopic với PhoBERT cho tiếng Việt — adapted cho Kaggle Dual T4.

    Thay đổi so với local version:
        1. Embedding: DualGPUEmbedder (split docs → GPU 0 & 1)
        2. OOM protection: batch_size tự giảm khi VRAM thấp
        3. Pre-compute embeddings trước fit_transform (tránh encode lại)
        4. free_gpu_memory() sau mỗi bước nặng

    Logic nghiệp vụ (export, schema, coherence) giữ nguyên 100%.
    """

    def __init__(
        self,
        embedding_model: str = "vinai/phobert-base",
        n_neighbors: int = 15,
        n_components: int = 5,
        min_dist: float = 0.0,
        min_cluster_size: int = 15,
        min_samples: int = 10,
        top_n_words: int = 10,
        verbose: bool = True,
        use_gpu: bool = True,
        # Kaggle-specific
        embedding_batch_size: int = 64,
        max_seq_length: int = 256,
    ):
        self.verbose = verbose

        # ── Device detection ─────────────────────────────────────────────────
        if use_gpu and torch.cuda.is_available():
            self.device = 'cuda'
            if self.verbose:
                for i in range(torch.cuda.device_count()):
                    print(f"🚀 GPU {i}: {torch.cuda.get_device_name(i)}")
        else:
            self.device = 'cpu'
            if self.verbose and use_gpu:
                print("⚠️  GPU not available, using CPU")

        # Save hyperparameters
        self.embedding_model_name  = embedding_model
        self.n_neighbors           = n_neighbors
        self.n_components          = n_components
        self.min_dist              = min_dist
        self.min_cluster_size      = min_cluster_size
        self.min_samples           = min_samples
        self.top_n_words           = top_n_words
        self.embedding_batch_size  = embedding_batch_size
        self.max_seq_length        = max_seq_length

        # ── [1] Dual-GPU Embedding model ─────────────────────────────────────
        if self.verbose:
            print(f"\n[1/4] Loading embedding model: {embedding_model}")

        self.embedding_model = DualGPUEmbedder(
            model_name=embedding_model,
            batch_size=embedding_batch_size,
            max_seq_length=max_seq_length,
            verbose=verbose,
        )

        if self.verbose:
            print(f"✅ Embedding model loaded")
            gpu_memory_summary()

        # ── [2] UMAP ─────────────────────────────────────────────────────────
        if self.verbose:
            print(f"\n[2/4] Configuring UMAP")
            print(f"  - n_neighbors : {n_neighbors}")
            print(f"  - n_components: {n_components}")
            print(f"  - min_dist    : {min_dist}")

        self.umap_model = UMAP(
            n_neighbors=n_neighbors,
            n_components=n_components,
            min_dist=min_dist,
            metric='cosine',
            random_state=42,
            low_memory=True,   # ← giảm RAM usage của UMAP
        )
        if self.verbose:
            print("✅ UMAP configured")

        # ── [3] HDBSCAN ──────────────────────────────────────────────────────
        if self.verbose:
            print(f"\n[3/4] Configuring HDBSCAN")
            print(f"  - min_cluster_size: {min_cluster_size}")
            print(f"  - min_samples     : {min_samples}")

        self.hdbscan_model = HDBSCAN(
            min_cluster_size=min_cluster_size,
            min_samples=min_samples,
            metric='euclidean',
            cluster_selection_method='eom',
            prediction_data=True,
            core_dist_n_jobs=-1,   # ← dùng tất cả CPU cores
        )
        if self.verbose:
            print("✅ HDBSCAN configured")

        # ── [4] BERTopic ─────────────────────────────────────────────────────
        if self.verbose:
            print(f"\n[4/4] Building BERTopic pipeline")

        self.topic_model = BERTopic(
            embedding_model=self.embedding_model,
            umap_model=self.umap_model,
            hdbscan_model=self.hdbscan_model,
            top_n_words=top_n_words,
            verbose=verbose,
            calculate_probabilities=True,
        )

        if self.verbose:
            print("✅ BERTopic pipeline ready")
            print(f"\n{'='*60}")
            print("Model initialized successfully!")
            print(f"{'='*60}\n")

        self.topics_ = None
        self.probs_  = None

    # =========================================================================
    # FIT — pre-compute embeddings → pass vào BERTopic (tránh encode 2 lần)
    # =========================================================================

    def fit(self, documents: List[str]) -> Tuple[List[int], np.ndarray]:
        """
        Train BERTopic trên documents.

        Kaggle optimization:
            Pre-compute embeddings bằng DualGPUEmbedder trước,
            sau đó truyền trực tiếp vào fit_transform để BERTopic
            KHÔNG encode lại → tránh double VRAM usage.
        """
        if self.verbose:
            print(f"\n{'='*60}")
            print("TRAINING BERTOPIC MODEL")
            print(f"{'='*60}")
            print(f"Documents: {len(documents):,}")
            print(f"Device   : {self.device}")

        if not documents:
            raise ValueError("documents cannot be empty")
        if not all(isinstance(doc, str) for doc in documents):
            raise ValueError("All documents must be strings")

        # ── Step 1: Pre-compute embeddings ────────────────────────────────
        if self.verbose:
            print("\n  [1/3] Pre-computing embeddings (Dual GPU)...")
            gpu_memory_summary()

        embeddings = self.embedding_model.encode(
            documents,
            batch_size=self.embedding_batch_size,
            show_progress_bar=True,
        )

        if self.verbose:
            print(f"  ✅ Embeddings shape: {embeddings.shape}")
            gpu_memory_summary()

        # Giải phóng GPU cache sau khi encode xong
        free_gpu_memory()

        # ── Step 2: BERTopic fit_transform với embeddings đã tính ─────────
        if self.verbose:
            print("\n  [2/3] Running BERTopic (UMAP + HDBSCAN + c-TF-IDF)...")

        topics, probs = self.topic_model.fit_transform(
            documents,
            embeddings=embeddings,  # ← KEY: truyền sẵn, không encode lại
        )

        free_gpu_memory()

        # ── Step 3: Store results ──────────────────────────────────────────
        self.topics_ = topics
        self.probs_  = probs

        if self.verbose:
            n_topics   = len(set(topics)) - (1 if -1 in topics else 0)
            n_outliers = sum(t == -1 for t in topics)
            print(f"\n{'='*60}")
            print("TRAINING COMPLETED")
            print(f"{'='*60}")
            print(f"✅ Topics found : {n_topics}")
            print(f"✅ Outliers     : {n_outliers} ({n_outliers/len(topics)*100:.1f}%)")
            print(f"✅ Assigned docs: {len(topics) - n_outliers:,}")
            print(f"{'='*60}\n")
            gpu_memory_summary()

        return topics, probs

    # =========================================================================
    # TRANSFORM (inference) — giữ nguyên logic
    # =========================================================================

    def transform(self, documents: List[str]) -> Tuple[List[int], np.ndarray]:
        if self.topics_ is None:
            raise ValueError(
                "Model chưa được train. Hãy gọi fit() trước hoặc load() model đã train."
            )
        if self.verbose:
            print(f"\nPredicting topics for {len(documents):,} documents...")

        # Pre-compute embeddings để tránh double encode
        embeddings = self.embedding_model.encode(
            documents,
            batch_size=self.embedding_batch_size,
            show_progress_bar=True,
        )
        free_gpu_memory()

        topics, probs = self.topic_model.transform(documents, embeddings=embeddings)
        free_gpu_memory()

        if self.verbose:
            print("✅ Prediction completed")
        return topics, probs

    # =========================================================================
    # ANALYSIS — giữ nguyên logic từ local version
    # =========================================================================

    def get_topic_info(self) -> pd.DataFrame:
        if self.topics_ is None:
            raise ValueError("Model chưa được train")
        return self.topic_model.get_topic_info()

    def get_topics(
        self, topic_id: Optional[int] = None
    ) -> Dict[int, List[Tuple[str, float]]]:
        if self.topics_ is None:
            raise ValueError("Model chưa được train")
        if topic_id is not None:
            topic = self.topic_model.get_topic(topic_id)
            return {topic_id: topic} if topic else {}
        all_topics = {}
        for tid in set(self.topics_):
            if tid != -1:
                topic = self.topic_model.get_topic(tid)
                if topic:
                    all_topics[tid] = topic
        return all_topics

    def calculate_coherence(
        self,
        documents: List[str],
        coherence_type: str = 'c_v',
    ) -> float:
        if self.topics_ is None:
            raise ValueError("Model chưa được train")
        if self.verbose:
            print(f"\nCalculating coherence ({coherence_type})...")
        topics_dict = self.get_topics()
        if not topics_dict:
            if self.verbose:
                print("⚠️  No topics found (all outliers)")
            return 0.0
        topics_words = [
            [word for word, _ in topics_dict[tid]]
            for tid in sorted(topics_dict.keys())
        ]
        texts = [doc.split() for doc in documents]
        dictionary = Dictionary(texts)
        coherence_model = CoherenceModel(
            topics=topics_words,
            texts=texts,
            dictionary=dictionary,
            coherence=coherence_type,
        )
        coherence_score = coherence_model.get_coherence()
        if self.verbose:
            print(f"✅ Coherence {coherence_type.upper()}: {coherence_score:.4f}")
        return coherence_score

    def get_summary(self) -> Dict[str, Any]:
        if self.topics_ is None:
            return {
                'status': 'not_trained',
                'device': self.device,
                'hyperparameters': {
                    'n_neighbors': self.n_neighbors,
                    'n_components': self.n_components,
                    'min_dist': self.min_dist,
                    'min_cluster_size': self.min_cluster_size,
                    'min_samples': self.min_samples,
                    'top_n_words': self.top_n_words,
                },
            }
        n_topics   = len(set(self.topics_)) - (1 if -1 in self.topics_ else 0)
        n_outliers = sum(t == -1 for t in self.topics_)
        return {
            'status': 'trained',
            'n_documents': len(self.topics_),
            'n_topics': n_topics,
            'n_outliers': n_outliers,
            'outlier_ratio': n_outliers / len(self.topics_),
            'hyperparameters': {
                'embedding_model': self.embedding_model_name,
                'n_neighbors': self.n_neighbors,
                'n_components': self.n_components,
                'min_dist': self.min_dist,
                'min_cluster_size': self.min_cluster_size,
                'min_samples': self.min_samples,
                'top_n_words': self.top_n_words,
            },
            'device': self.device,
        }

    def visualize_topics(self):
        if self.topics_ is None:
            raise ValueError("Model chưa được train")
        return self.topic_model.visualize_topics()

    # =========================================================================
    # SCHEMA-COMPLIANT EXPORT — giữ nguyên 100% logic từ local version
    # =========================================================================

    def export_post_topics(
        self,
        post_ids: List[str],
        output_path: str,
        coherence_score: Optional[float] = None,
    ) -> pd.DataFrame:
        if self.topics_ is None:
            raise ValueError("Model chưa được train. Hãy gọi fit() trước.")
        if len(post_ids) != len(self.topics_):
            raise ValueError(
                f"Số lượng post_ids ({len(post_ids)}) "
                f"không khớp với số topics ({len(self.topics_)})."
            )
        predicted_at  = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
        probs_array   = np.array(self.probs_) if self.probs_ is not None else None
        rows = []
        for i, (pid, tid) in enumerate(zip(post_ids, self.topics_)):
            if probs_array is not None and probs_array.ndim == 2:
                prob = float(probs_array[i, tid]) if 0 <= tid < probs_array.shape[1] else 0.0
            elif probs_array is not None and probs_array.ndim == 1:
                prob = float(probs_array[i])
            else:
                prob = 0.0
            rows.append({
                "post_id": str(pid),
                "topic_id": int(tid),
                "topic_probability": round(prob, 6),
                "model_type": "bertopic",
                "predicted_at": predicted_at,
            })
        df = pd.DataFrame(rows, columns=["post_id","topic_id","topic_probability","model_type","predicted_at"])
        df["topic_id"]          = df["topic_id"].astype("int32")
        df["topic_probability"] = df["topic_probability"].astype("float32")
        os.makedirs(os.path.dirname(output_path) if os.path.dirname(output_path) else ".", exist_ok=True)
        df.to_csv(output_path, index=False, encoding="utf-8")
        if self.verbose:
            print(f"✅ stg_post_topics → {output_path} ({len(df):,} rows)")
            if coherence_score is not None:
                print(f"   coherence_score (info only): {coherence_score:.4f}")
        return df

    def export_stg_topics(
        self,
        output_path: str,
        coherence_score: Optional[float] = None,
        model_version: str = "bertopic_v2",
    ) -> pd.DataFrame:
        if self.topics_ is None:
            raise ValueError("Model chưa được train. Hãy gọi fit() trước.")
        created_at  = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
        topics_dict = self.get_topics()
        rows = []
        for tid in sorted(topics_dict.keys()):
            word_score_list = topics_dict[tid]
            top_keywords    = [w for w, _ in word_score_list[:10]]
            label           = " | ".join(top_keywords[:3]) if top_keywords else f"topic_{tid}"
            rows.append({
                "topic_id": int(tid),
                "label": label,
                "top_keywords": top_keywords,
                "coherence_score": float(coherence_score) if coherence_score is not None else None,
                "model_version": model_version,
                "created_at": created_at,
            })
        df = pd.DataFrame(
            rows,
            columns=["topic_id","label","top_keywords","coherence_score","model_version","created_at"],
        )
        df["topic_id"] = df["topic_id"].astype("int32")
        if coherence_score is not None:
            df["coherence_score"] = df["coherence_score"].astype("float32")
        os.makedirs(os.path.dirname(output_path) if os.path.dirname(output_path) else ".", exist_ok=True)
        csv_rows = df.copy()
        csv_rows["top_keywords"] = csv_rows["top_keywords"].apply(
            lambda kws: json.dumps(kws, ensure_ascii=False)
        )
        csv_rows.to_csv(output_path, index=False, encoding="utf-8")
        if self.verbose:
            print(f"✅ stg_topics → {output_path} ({len(df):,} topics)")
        return df

    # =========================================================================
    # SAVE / LOAD — giữ nguyên logic
    # =========================================================================

    def save(self, path: str) -> None:
        if self.verbose:
            print(f"\nSaving model to: {path}")
        os.makedirs(path, exist_ok=True)

        bertopic_path = os.path.join(path, "bertopic_model")
        self.topic_model.save(bertopic_path, serialization="pickle")
        if self.verbose:
            print(f"✅ BERTopic model saved to {bertopic_path}")

        config = {
            'embedding_model': self.embedding_model_name,
            'n_neighbors': self.n_neighbors,
            'n_components': self.n_components,
            'min_dist': self.min_dist,
            'min_cluster_size': self.min_cluster_size,
            'min_samples': self.min_samples,
            'top_n_words': self.top_n_words,
            'device': self.device,
            'embedding_batch_size': self.embedding_batch_size,
            'max_seq_length': self.max_seq_length,
        }
        config_path = os.path.join(path, "config.pkl")
        with open(config_path, 'wb') as f:
            pickle.dump(config, f)
        if self.verbose:
            print(f"✅ Config saved to {config_path}")

        if self.topics_ is not None:
            results_path = os.path.join(path, "topics.pkl")
            with open(results_path, 'wb') as f:
                pickle.dump({'topics': self.topics_, 'probs': self.probs_}, f)
            if self.verbose:
                print(f"✅ Topics/probs saved to {results_path}")

            self.export_stg_topics(
                output_path=os.path.join(path, "stg_topics.csv"),
                model_version="bertopic_v2",
            )
            fake_ids = [f"doc_{i}" for i in range(len(self.topics_))]
            self.export_post_topics(
                post_ids=fake_ids,
                output_path=os.path.join(path, "stg_post_topics.csv"),
            )
            if self.verbose:
                print(
                    "⚠️  stg_post_topics.csv sử dụng index giả (doc_0, doc_1, ...). "
                    "Gọi export_post_topics(real_post_ids, ...) để ghi lại với post_ids thực."
                )

        if self.verbose:
            print(f"\n{'='*60}")
            print("Model saved successfully!")
            print(f"{'='*60}\n")

    @classmethod
    def load(cls, path: str, verbose: bool = True) -> 'VietnameseBERTopicModel':
        if verbose:
            print(f"\nLoading model from: {path}")
        config_path = os.path.join(path, "config.pkl")
        with open(config_path, 'rb') as f:
            config = pickle.load(f)
        if verbose:
            print("✅ Config loaded")

        instance = cls(
            embedding_model=config['embedding_model'],
            n_neighbors=config['n_neighbors'],
            n_components=config['n_components'],
            min_dist=config['min_dist'],
            min_cluster_size=config['min_cluster_size'],
            min_samples=config['min_samples'],
            top_n_words=config['top_n_words'],
            verbose=verbose,
            embedding_batch_size=config.get('embedding_batch_size', 64),
            max_seq_length=config.get('max_seq_length', 256),
        )

        bertopic_path = os.path.join(path, "bertopic_model")
        instance.topic_model = BERTopic.load(bertopic_path)
        if verbose:
            print("✅ BERTopic model loaded")

        results_path = os.path.join(path, "topics.pkl")
        if os.path.exists(results_path):
            with open(results_path, 'rb') as f:
                results = pickle.load(f)
            instance.topics_ = results['topics']
            instance.probs_  = results['probs']
            if verbose:
                print("✅ Topics/probs loaded")

        if verbose:
            print(f"\n{'='*60}")
            print("Model loaded successfully!")
            print(f"{'='*60}\n")
        return instance


# ── Helper: batch training (giữ nguyên logic) ─────────────────────────────────
def train_bertopic_batch(
    documents: List[str],
    batch_size: int = 1000,
    **kwargs,
) -> VietnameseBERTopicModel:
    if len(documents) <= batch_size:
        model = VietnameseBERTopicModel(**kwargs)
        model.fit(documents)
        return model
    print(f"⚠️  Large dataset ({len(documents):,} docs)")
    print(f"   Training on full dataset (BERTopic handles it well)\n")
    model = VietnameseBERTopicModel(**kwargs)
    model.fit(documents)
    return model


print("✅ VietnameseBERTopicModel (Dual T4) defined")

## 4. Load dữ liệu

Thay `documents` bằng list clean_text thực tế của bạn.  
Nếu data từ CSV trên Kaggle Dataset, dùng `/kaggle/input/<dataset-slug>/...`

In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────────
# OPTION A: từ CSV trong Kaggle Dataset
# df = pd.read_csv("/kaggle/input/your-dataset/clean_posts.csv")
# documents = df["clean_text"].dropna().tolist()

# OPTION B: dummy data để test pipeline
documents = [
    "điện thoại iphone pin trâu camera đẹp",
    "samsung galaxy màn hình amoled chất lượng cao",
    "laptop gaming rtx card đồ họa mạnh",
    "tai nghe bluetooth âm thanh bass mạnh",
    "smartwatch theo dõi sức khỏe nhịp tim",
] * 200  # nhân để có đủ docs cho HDBSCAN

print(f"Tổng số documents: {len(documents):,}")
print(f"Ví dụ: {documents[0][:80]}...")

## 5. Train model

In [ ]:
# ── Config theo dataset size ───────────────────────────────────────────────────
# embedding_batch_size:
#   - Dataset < 10k docs  → 128
#   - Dataset 10k–50k     → 64  (default)
#   - Dataset > 50k docs  → 32
#   - Nếu vẫn OOM        → 16 hoặc 8

model = VietnameseBERTopicModel(
    embedding_model="vinai/phobert-base",
    n_neighbors=15,
    n_components=5,
    min_dist=0.0,
    min_cluster_size=15,
    min_samples=10,
    top_n_words=10,
    verbose=True,
    use_gpu=True,
    # ── Kaggle-specific ──
    embedding_batch_size=64,   # ← giảm nếu OOM
    max_seq_length=256,        # ← giảm nếu OOM (PhoBERT max=256)
)

topics, probs = model.fit(documents)
print("\nSummary:")
print(model.get_summary())

## 6. Phân tích kết quả

In [ ]:
topic_info = model.get_topic_info()
print(f"Topic info ({len(topic_info)} topics):")
topic_info.head(20)

In [ ]:
# Top words mỗi topic
all_topics = model.get_topics()
for tid, words in list(all_topics.items())[:5]:
    print(f"Topic {tid}: {[w for w, _ in words[:5]]}")

In [ ]:
# Coherence score
coherence_score = model.calculate_coherence(documents, coherence_type='c_v')
print(f"Coherence C_V: {coherence_score:.4f}")

## 7. Export CSV theo schema

In [ ]:
OUTPUT_DIR = "/kaggle/working/bertopic_output"

# stg_topics
df_topics = model.export_stg_topics(
    output_path=os.path.join(OUTPUT_DIR, "stg_topics.csv"),
    coherence_score=coherence_score,
    model_version="bertopic_v2",
)
df_topics.head()

In [ ]:
# stg_post_topics — thay bằng post_ids thực
post_ids = [f"post_{i}" for i in range(len(documents))]  # ← thay bằng real IDs

df_post_topics = model.export_post_topics(
    post_ids=post_ids,
    output_path=os.path.join(OUTPUT_DIR, "stg_post_topics.csv"),
    coherence_score=coherence_score,
)
df_post_topics.head()

## 8. Save model

In [ ]:
model.save("/kaggle/working/bertopic_model/")
print("\nFiles saved:")
for f in os.listdir("/kaggle/working/bertopic_model/"):
    print(f"  {f}")

## 9. (Optional) Visualize

In [ ]:
# Interactive visualization (Plotly)
# fig = model.visualize_topics()
# fig.write_html("/kaggle/working/topics_viz.html")
# fig.show()
print("Uncomment cell trên để visualize.")

## 10. OOM Troubleshooting

| Triệu chứng | Fix |
|---|---|
| OOM khi encoding | Giảm `embedding_batch_size` (64 → 32 → 16) |
| OOM với seq dài | Giảm `max_seq_length` (256 → 128) |
| OOM sau UMAP | Tăng `low_memory=True` (đã bật), giảm `n_components` |
| RAM hết (UMAP/HDBSCAN) | Subsample documents (50k là reasonable) |
| Notebook crash khi load model | `free_gpu_memory()` trước khi load |